In [12]:
import pandas as pd
import os
from glob import glob
import numpy as np

class DataFormatter:
    def __init__(self):
        self.files_processed = []
    
    def prepare_files_for_labeling(self, data_directory="./"):
        """
        Step 1: Add necessary columns to all CSV files for labeling
        """
        print("STEP 1: Preparing files for labeling")
        print("="*60)
        
        csv_files = glob(os.path.join(data_directory, "*.csv"))
        csv_files = [f for f in csv_files if 'backup' not in f and 'labeled' not in f]
        
        print(f"Found {len(csv_files)} files to prepare\n")
        
        for file_path in csv_files:
            try:
                dealership_name = os.path.basename(file_path).replace('.csv', '')
                
                # Load file
                df = pd.read_csv(file_path, encoding='unicode_escape')
                
                # Check if wiI7pd column exists
                if 'wiI7pd' not in df.columns:
                    print(f"⚠ {dealership_name}: 'wiI7pd' column not found, skipping...")
                    continue
                
                # Add necessary columns if they don't exist
                if 'dealership_name' not in df.columns:
                    df['dealership_name'] = dealership_name
                
                if 'manual_label' not in df.columns:
                    df['manual_label'] = ''  # Empty, ready for labeling
                
                if 'review_id' not in df.columns:
                    df['review_id'] = range(1, len(df) + 1)
                
                # Remove any rows where review text is null or too short
                df = df[df['wiI7pd'].notna()]
                df = df[df['wiI7pd'].astype(str).str.len() > 10]
                
                # Save prepared file
                output_file = file_path.replace('.csv', '_ready_for_labeling.csv')
                df.to_csv(output_file, index=False)
                
                self.files_processed.append({
                    'original': file_path,
                    'prepared': output_file,
                    'dealership': dealership_name,
                    'total_reviews': len(df),
                    'already_labeled': len(df[df['manual_label'] != ''])
                })
                
                print(f"✓ {dealership_name}")
                print(f"  Total reviews: {len(df)}")
                print(f"  Output: {output_file}\n")
                
            except Exception as e:
                print(f"✗ Error processing {file_path}: {e}\n")
        
        self.show_preparation_summary()
        return self.files_processed
    
    def show_preparation_summary(self):
        """Show summary of prepared files"""
        print("\n" + "="*60)
        print("PREPARATION SUMMARY")
        print("="*60)
        
        total_reviews = sum(f['total_reviews'] for f in self.files_processed)
        total_labeled = sum(f['already_labeled'] for f in self.files_processed)
        
        print(f"Files prepared: {len(self.files_processed)}")
        print(f"Total reviews: {total_reviews}")
        print(f"Already labeled: {total_labeled}")
        print(f"Remaining to label: {total_reviews - total_labeled}")
        
        print("\nFiles ready for labeling:")
        for f in self.files_processed:
            status = f"({f['already_labeled']}/{f['total_reviews']} labeled)" if f['already_labeled'] > 0 else "(not started)"
            print(f"  - {f['prepared']} {status}")
    
    def combine_labeled_files(self, data_directory="./labelled/", output_file="master_training_data.csv"):
        """
        Step 2: Combine all labeled files into one master training file
        """
        print("\nSTEP 2: Combining labeled files")
        print("="*60)
        
        # Find all labeled files
        labeled_files = glob(os.path.join(data_directory, "*_ready_for_labeling.csv"))
        
        if not labeled_files:
            print("No labeled files found!")
            return None
        
        combined_data = []
        total_labeled = 0
        total_reviews = 0
        
        for file_path in labeled_files:
            try:
                df = pd.read_csv(file_path, encoding='unicode_escape')
                
                # Only include rows that have been labeled
                labeled_df = df[df['manual_label'].isin([0, 1, '0', '1', 'positive', 'negative'])]
                
                if len(labeled_df) > 0:
                    dealership_name = os.path.basename(file_path).replace('_ready_for_labeling.csv', '')
                    
                    # Standardize labels to 0/1
                    labeled_df['manual_label'] = labeled_df['manual_label'].apply(self.standardize_label)
                    
                    # Keep only necessary columns
                    final_df = labeled_df[['wiI7pd', 'dealership_name', 'manual_label']].copy()
                    final_df.columns = ['review_text', 'dealership_name', 'label']
                    
                    combined_data.append(final_df)
                    
                    total_labeled += len(labeled_df)
                    total_reviews += len(df)
                    
                    print(f"✓ {dealership_name}: {len(labeled_df)} labeled reviews")
                
            except Exception as e:
                print(f"✗ Error processing {file_path}: {e}")
        
        if not combined_data:
            print("\nNo labeled data found! Please label some reviews first.")
            return None
        
        # Combine all dataframes
        master_df = pd.concat(combined_data, ignore_index=True)
        
        # Shuffle the data
        master_df = master_df.sample(frac=1, random_state=42).reset_index(drop=True)
        
        # Save combined file
        master_df.to_csv(output_file, index=False)
        
        print(f"\n{'='*60}")
        print("COMBINATION SUMMARY")
        print("="*60)
        print(f"Total files processed: {len(labeled_files)}")
        print(f"Total reviews across all files: {total_reviews}")
        print(f"Total labeled reviews: {total_labeled}")
        print(f"Labeling progress: {(total_labeled/total_reviews)*100:.1f}%")
        print(f"\nLabel distribution:")
        print(master_df['label'].value_counts())
        print(f"\nDealership distribution:")
        print(master_df['dealership_name'].value_counts())
        print(f"\n✓ Master training file saved: {output_file}")
        
        return master_df
    
    def standardize_label(self, label):
        """Convert various label formats to 0/1"""
        if label in [1, '1', 'positive', 'Positive', 'POSITIVE']:
            return 1
        elif label in [0, '0', 'negative', 'Negative', 'NEGATIVE']:
            return 0
        else:
            return None
    
    def check_data_quality(self, master_file="master_training_data.csv"):
        """
        Step 3: Check quality of combined data
        """
        print("\nSTEP 3: Data Quality Check")
        print("="*60)
        
        try:
            df = pd.read_csv(master_file)
            
            print(f"Total samples: {len(df)}")
            print(f"\nColumns: {list(df.columns)}")
            
            # Check for missing values
            print(f"\nMissing values:")
            print(df.isnull().sum())
            
            # Check label distribution
            print(f"\nLabel distribution:")
            label_counts = df['label'].value_counts()
            print(label_counts)
            
            positive_pct = (label_counts.get(1, 0) / len(df)) * 100
            negative_pct = (label_counts.get(0, 0) / len(df)) * 100
            print(f"\nPositive: {positive_pct:.1f}%")
            print(f"Negative: {negative_pct:.1f}%")
            
            # Check for class imbalance
            if positive_pct < 30 or positive_pct > 70:
                print(f"\n⚠ Warning: Class imbalance detected!")
                print(f"Consider balancing your labels for better model performance")
            
            # Check reviews per dealership
            print(f"\nReviews per dealership:")
            dealership_counts = df['dealership_name'].value_counts()
            print(dealership_counts)
            
            # Check minimum reviews per dealership
            min_reviews = dealership_counts.min()
            if min_reviews < 20:
                print(f"\n⚠ Warning: Some dealerships have fewer than 20 reviews")
                print(f"Minimum: {min_reviews} reviews")
                print(f"This might affect LSTM sequence creation")
            
            # Text length statistics
            df['text_length'] = df['review_text'].astype(str).str.len()
            print(f"\nReview length statistics:")
            print(f"Mean: {df['text_length'].mean():.0f} characters")
            print(f"Median: {df['text_length'].median():.0f} characters")
            print(f"Min: {df['text_length'].min()} characters")
            print(f"Max: {df['text_length'].max()} characters")
            
            # Sample reviews
            print(f"\n{'='*60}")
            print("SAMPLE REVIEWS")
            print("="*60)
            
            for label in [0, 1]:
                print(f"\nLabel {label} ({'Negative/Untrustworthy' if label == 0 else 'Positive/Trustworthy'}):")
                samples = df[df['label'] == label].head(2)
                for idx, row in samples.iterrows():
                    text = row['review_text'][:100] + "..." if len(str(row['review_text'])) > 100 else row['review_text']
                    print(f"  - {text}")
            
            print(f"\n{'='*60}")
            print("✓ Data quality check complete!")
            
            # Assessment
            print(f"\nASSESSMENT:")
            if len(df) >= 500:
                print("✓ Sufficient data for training (500+ samples)")
            else:
                print(f"⚠ Limited data ({len(df)} samples). Aim for 500+ for best results")
            
            if 30 <= positive_pct <= 70:
                print("✓ Good label balance")
            else:
                print("⚠ Consider balancing your labels")
            
            if min_reviews >= 20:
                print("✓ All dealerships have sufficient reviews")
            else:
                print(f"⚠ Some dealerships need more labeled reviews")
            
            return True
            
        except FileNotFoundError:
            print(f"Error: {master_file} not found!")
            print("Please combine labeled files first using combine_labeled_files()")
            return False
        except Exception as e:
            print(f"Error during quality check: {e}")
            return False
    
    def create_train_test_split(self, master_file="master_training_data.csv", test_size=0.3):
        """
        Step 4: Split data into train and test sets
        """
        print(f"\nSTEP 4: Creating train/test split ({int((1-test_size)*100)}/{int(test_size*100)})")
        print("="*60)
        
        try:
            df = pd.read_csv(master_file)
            
            from sklearn.model_selection import train_test_split
            
            # Stratified split to maintain label distribution
            train_df, test_df = train_test_split(
                df, 
                test_size=test_size, 
                random_state=42,
                stratify=df['label']
            )
            
            # Save splits
            train_df.to_csv('training_data.csv', index=False)
            test_df.to_csv('testing_data.csv', index=False)
            
            print(f"Training set: {len(train_df)} samples")
            print(f"  Positive: {len(train_df[train_df['label']==1])}")
            print(f"  Negative: {len(train_df[train_df['label']==0])}")
            
            print(f"\nTesting set: {len(test_df)} samples")
            print(f"  Positive: {len(test_df[test_df['label']==1])}")
            print(f"  Negative: {len(test_df[test_df['label']==0])}")
            
            print(f"\n✓ Files saved:")
            print(f"  - training_data.csv")
            print(f"  - testing_data.csv")
            
            return train_df, test_df
            
        except Exception as e:
            print(f"Error creating splits: {e}")
            return None, None

# Easy-to-use workflow functions
def workflow_step_1_prepare():
    """Step 1: Prepare all files for labeling"""
    formatter = DataFormatter()
    formatter.prepare_files_for_labeling()
    print("\n✓ NEXT STEP: Label the reviews in the *_ready_for_labeling.csv files")

def workflow_step_2_combine():
    """Step 2: Combine labeled files"""
    formatter = DataFormatter()
    master_df = formatter.combine_labeled_files()
    if master_df is not None:
        print("\n✓ NEXT STEP: Check data quality")
    return master_df

def workflow_step_3_check():
    """Step 3: Check data quality"""
    formatter = DataFormatter()
    formatter.check_data_quality()
    print("\n✓ NEXT STEP: Create train/test split")

def workflow_step_4_split():
    """Step 4: Create train/test split"""
    formatter = DataFormatter()
    train_df, test_df = formatter.create_train_test_split()
    if train_df is not None:
        print("\n✓ READY FOR MODEL TRAINING!")
    return train_df, test_df

# Complete workflow
def complete_data_preparation_workflow():
    """Run all steps in sequence"""
    print("SMART TRUST METER - DATA PREPARATION WORKFLOW")
    print("="*60)
    
    formatter = DataFormatter()
    
    # Step 1: Prepare files
    print("\n>>> Running Step 1: Preparing files...")
    formatter.prepare_files_for_labeling()
    
    print("\n" + "="*60)
    print("MANUAL LABELING REQUIRED")
    print("="*60)
    print("Please label the reviews in the generated files")
    print("Then run workflow_step_2_combine() to continue")

if __name__ == "__main__":
    # Run step 1
    # workflow_step_1_prepare()
    # workflow_step_2_combine()
    # workflow_step_3_check()
    workflow_step_4_split()


STEP 4: Creating train/test split (70/30)
Training set: 718 samples
  Positive: 467
  Negative: 251

Testing set: 309 samples
  Positive: 201
  Negative: 108

✓ Files saved:
  - training_data.csv
  - testing_data.csv

✓ READY FOR MODEL TRAINING!
